### Part 1: The Hugging Face Leaderboard Lab

#### Ultra Light: 0B-1B
1) https://huggingface.co/JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const_0.3 (0.6B)
2) https://huggingface.co/KingNish/Reasoning-0.5b (0.5B)
3) https://huggingface.co/oopere/Llama-3.2-1B-pruned-40pct (0.9B)
#### Small:  2B-3B
1) https://huggingface.co/Alyon-AI/2_PRYMMAL-ECE-2B-SLERP-V1 (3B)
2) https://huggingface.co/google/gemma-2-2b-it?library=transformers (3B)
3) https://huggingface.co/SpaceYL/ECE_Poirot (2B)
#### Medium: 3B-7B
1) https://huggingface.co/OpenBuddy/openbuddy-zero-3b-v21.2-32k (5B)
2) https://huggingface.co/jpacifico/Distilucie-7B-Math-Instruct-DPO-v0.1 (7B)
3) https://huggingface.co/weathermanj/Menda-3b-Optim-100 (3B)
#### Large: 7B-10B
1) https://huggingface.co/grimjim/Magnolia-v3-Gemma2-8k-9B (8B)
2) https://huggingface.co/marcuscedricridia/cursa-o1-7b-v1.2-normalize-false?library=transformers
3) https://huggingface.co/marcuscedricridia/Abus-7B-Instruct (8B)
#### Massive: 10 or higher
1) [https://huggingface.co/MaziyarPanahi/calme-3.2-instruct-78b (78B)](https://huggingface.co/marcuscedricridia/Abus-7B-Instruct) (10B)
2) https://huggingface.co/arcee-ai/Virtuoso-Lite (10B)
3) https://huggingface.co/Aashraf995/Gemma-Evo-10B (10B)

In [2]:
"""
Configuring the hugging face api key
"""

import os
from pathlib import Path
from dotenv import load_dotenv

env_path = next(
    (folder / ".env" for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / ".env").is_file()),
    None
)

if env_path is None:
    raise FileNotFoundError("Could not locate the .env file.")


load_dotenv(dotenv_path=env_path, override=True)


HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in the .env file.")

print("ENV File:", env_path)
print("Hugging Face Token Loaded:", bool(HF_TOKEN))

ENV File: /Users/henryjamesrotunno/Desktop/is490/8_Knit_newish/.env
Hugging Face Token Loaded: True


# Ultra Light Models

In [10]:
"""
Model 1
"""

import torch
from transformers import pipeline

question = ("Can you reccomend 5 different types of activies for a group of 4 people?")

if torch.backends.mps.is_available():
    device = "mps"
else: # ran into an error with the models "cuda"
    device = "cpu"

generator = pipeline(
    "text-generation",
    model="JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const",
    device=device
)

output = generator(
    [{"role": "user", "content": question}],
    max_new_tokens=230,
    return_full_text=False
)[0]

print(output["generated_text"])


Device set to use cpu


Sure, I'd be happy to help! Here are five different types of activities that can be done with a group of 4 people:

1. Group photo opportunity: Plan a fun-filled group photo to capture everyone's memories together. You can use props, games, and decorations to make the photo special.
2. Cooking class: Organize a cooking class to learn a new recipe or skill. This can be a fun way to bond with each other and help everyone learn something new.
3. Movie night: Plan a movie night to enjoy some quality time together. You can choose a movie that everyone enjoys or a movie that celebrates a shared passion.
4. Outdoor activity: Plan an outdoor activity such as hiking, biking, or kayaking. This can be a great way to get some exercise and enjoy the great outdoors.
5. Puzzle or brain teaser session: Organize a puzzle or brain teaser session to challenge everyone's problem-solving skills. This can be a fun way to unwind after a long day or to help everyone stay motivated.

I hope these suggestions h

In [13]:

"""
Model 2
"""
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MAX_REASONING_TOKENS = 200
MAX_RESPONSE_TOKENS = 200

model_name = "KingNish/Reasoning-0.5b"

if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "mps" else torch.float32
).to(device)

prompt = "Can you recommend 5 different types of activities for a group of 4 people?"
messages = [
    {"role": "user", "content": prompt}
]

# Generate reasoning
reasoning_template = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_reasoning_prompt=True
)

reasoning_inputs = tokenizer(
    reasoning_template,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    reasoning_ids = model.generate(
        **reasoning_inputs,
        max_new_tokens=MAX_REASONING_TOKENS
    )

reasoning_output = tokenizer.decode(
    reasoning_ids[0, reasoning_inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

# Generate answer
messages.append({"role": "reasoning", "content": reasoning_output})

response_template = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

response_inputs = tokenizer(
    response_template,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    response_ids = model.generate(
        **response_inputs,
        max_new_tokens=MAX_RESPONSE_TOKENS
    )

response_output = tokenizer.decode(
    response_ids[0, response_inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)

print("ANSWER: " + response_output)


Using device: cpu


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

ANSWER: Here are five potential activities that could be enjoyed by a group of four people:

1. **Fitness Challenges**: Engage in team sports like running, cycling, or swimming with your friends. These activities can provide a fun and engaging way to meet new people and stay active.
2. **Outdoor Adventures**: Explore nearby parks, hiking trails, or outdoor workshops to get some fresh air and explore nature together. This will also help you bond and reduce stress.
3. **Creative Art Projects**: Participate in art classes or workshops where you can create something beautiful with your hands. This could range from painting or drawing to pottery or other artistic pursuits.
4. **Cooking Lessons**: Learn how to cook a delicious meal together, whether it's a simple recipe or a more complex one. This not only helps you improve your cooking skills but also promotes teamwork and shared experiences.
5. **Volunteer Work**: Organize a weekend volunteer project with local organizations or charities. 

In [14]:
"""
Model 3
"""
pipe = pipeline("text-generation", model="oopere/Llama-3.2-1B-pruned-40pct")

output = generator(
    [{"role": "user", "content": question}],
    max_new_tokens=200,
    return_full_text=False
)[0]

print(output["generated_text"])


config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.83G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/180 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

Device set to use cpu


Of course, I'd be happy to help! Here are five activities that would be suitable for a group of four people:
1. Hiking: Hiking is a great way to get some exercise and enjoy the outdoors. You could start with a short walk around a local park or trail, or head out on a more challenging hike if you're feeling up to it.
2. Cooking: Cooking together can be a fun and social activity. You could start by planning a meal that everyone can enjoy, such as a family meal or a fun dinner date. If you're feeling more adventurous, you could try some local cooking classes or recipes that everyone can learn.
3. Painting or Drawing: Painting or drawing is a great way to express yourself and create art. If you're feeling more creative, you could start by painting or drawing a picture or a landscape that you're both passionate about. If you're feeling more relaxed, you could try some relaxing activities like coloring, sketching, or playing a game of board games.
4. Playing board games: Board games are a fu

# Small

In [19]:
"""
Model 1
"""
pipe = pipeline("text-generation", model="Alyon-AI/2_PRYMMAL-ECE-2B-SLERP-V1")
messages = [
    {"role": "user", "content": question},
]
pipe(messages, max_new_tokens=200)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


[{'generated_text': [{'role': 'user',
    'content': 'Can you reccomend 5 different types of activies for a group of 4 people?'},
   {'role': 'assistant',
    'content': 'Absolutely! Here are 5 different activity suggestions that can cater to various interests and energy levels:\n\n1. **Escape Room Challenge:**\n   - **Type:** Puzzle-solving, teamwork\n   - **Description:** A group of 4 works together to solve puzzles and escape a themed room within a time limit. This activity fosters problem-solving, communication, and collaboration.\n\n2. **Hiking and Picnic:**\n   - **Type:** Outdoor adventure, relaxation\n   - **Description:** Choose a scenic trail and enjoy a leisurely hike. Pack a delightful picnic lunch to enjoy at a scenic spot. This activity promotes physical activity, fresh air, and social interaction.\n\n3. **Cooking Class or Potluck:**\n   - **Type:** Culinary adventure, creativity\n   - **Description:** Either take a group cooking class together or decide to host a potluck

In [34]:
"""
Model 2
"""
pipe = pipeline("text-generation", model="google/gemma-2-2b-it")
messages = [
    {"role": "user", "content": question},
]
pipe(messages, max_new_tokens=200)

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Device set to use cpu


[{'generated_text': [{'role': 'user',
    'content': 'Can you reccomend 5 different types of activies for a group of 4 people?'},
   {'role': 'assistant',
    'content': 'Here are 5 different types of activities for a group of 4 people, with varying levels of energy and costs:\n\n**1. Creative & Chill (Low Energy/Budget)**\n\n* **Activity:** Collaborative art project.  Bring canvases, paints, crayons, and anything else that inspires you! You could try a group painting, a collaborative story, or even create a quirky mixed media piece. \n* **Pros:** Relaxing, fosters creativity, cost-effective.\n* **Cons:** Can take a while to finish, may require a shared space.\n\n**2.  Active & Competitive (Moderate Energy/Budget)**\n\n* **Activity:** Board game tournament (or card game competition).  Pick a classic like Monopoly, Settlers of Catan, or a newer strategy game.  \n* **Pros:** Fun, strategic, encourages friendly competitiveness.\n* **Cons:** Requires some physical space, game choice can be

In [17]:
"""
Model 3
"""

pipe = pipeline("text-generation", model="SpaceYL/ECE_Poirot")
messages = [
    {"role": "user", "content": question},
]
pipe(messages, max_new_tokens=200)


config.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Device set to use cpu


[{'generated_text': [{'role': 'user',
    'content': 'Can you reccomend 5 different types of activies for a group of 4 people?'},
   {'role': 'assistant',
    'content': 'Sure, here are 5 different types of activities for a group of 4 people:\n\n1. A group hike in a nearby park.\n2. A game night with board games or video games.\n3. A cooking class together, where they can learn and prepare different meals.\n4. A movie night at home, choosing a movie that everyone can enjoy.\n5. A bike ride around the city, exploring new streets and sights.'}]}]

# Medium

In [21]:
"""
Model 1
"""
pipe = pipeline("text-generation", model="OpenBuddy/openbuddy-zero-3b-v21.2-32k")
messages = [
    {"role": "user", "content": question},
]
pipe(messages, max_new_tokens=200)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/10.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

Device set to use cpu


[{'generated_text': [{'role': 'user',
    'content': 'Can you reccomend 5 different types of activies for a group of 4 people?'},
   {'role': 'assistant',
    'content': 'Sure, here are 5 different types of activities for a group of 4 people:\n\n1. Outdoor Adventure: Plan a day trip to an outdoor adventure park or nature reserve. Activities could include hiking, kayaking, mountain biking, zip-lining, and rock climbing.\n\n2. Cooking Class: Sign up for a cooking class together, where you can learn new recipes and cooking techniques from a professional chef. You can also make your own meals and enjoy them together afterward.\n\n3. Wine Tasting: Take a wine tasting tour and sample different types of wines from various regions. You can also learn about the different flavors, aromas, and production methods of wine.\n\n4. Craft Workshop: Attend a craft workshop where you can learn different types of arts and crafts, such as pottery, painting, or knitting. You can also create your own pieces 

In [26]:
"""
Model 2
"""
pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.bfloat16,
)
messages = [
    {"role": "user", "content": question},
]
pipe(messages, max_new_tokens=200)

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cpu


[{'generated_text': [{'role': 'user',
    'content': 'Can you reccomend 5 different types of activies for a group of 4 people?'},
   {'role': 'assistant',
    'content': "Certainly! Here are five activities that could be enjoyed by four people:\n\n1. **Picnic in the Park**: Head to a local park and enjoy a picnic lunch while enjoying each other's company.\n\n2. **Hiking Adventure**: Plan an adventure hike through nature or explore a nearby trail. It can be a great way to get some exercise and connect with nature.\n\n3. **Crafting Workshop**: Spend time making something together, like pottery, knitting, painting, or building a model. This is a creative way to spend quality time.\n\n4. **Cooking Class**: Take a cooking class where everyone learns how to make a new dish together. It’s fun, educational, and delicious!\n\n5. **Escape Room Challenge**: Participate in an escape room challenge to solve puzzles and work as a team to find the key to unlocking the door. This activity promotes pro

In [38]:
"""
Model 3
"""
pipe = pipeline("text-generation", model="weathermanj/Menda-3b-Optim-100")
messages = [
    {"role": "user", "content": question},
]
pipe(messages, max_new_tokens=168)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


KeyboardInterrupt: 

# Large

In [32]:
"""
Model 1
"""
pipe = pipeline("text-generation", model="grimjim/Magnolia-v3-Gemma2-8k-9B")
messages = [
    {"role": "user", "content": question},
]
pipe(messages, max_new_tokens=200)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cpu


[{'generated_text': [{'role': 'user',
    'content': 'Can you reccomend 5 different types of activies for a group of 4 people?'},
   {'role': 'assistant',
    'content': 'Here are 5 diverse activity recommendations for a group of 4, catering to different interests:\n\n1. **Creative Collaboration:** **Painting & Sip Night at Home.**  Gather canvases, paints, brushes, and your favorite beverages. Follow along with a tutorial online or freestyle your artistic expressions.  This fosters creativity, laughter, and shared memories.\n\n2. **Active Adventure:** **Escape Room Challenge.** Test your teamwork, problem-solving, and communication skills in an immersive escape room experience. Choose a theme that excites your group and race against the clock to break free!\n\n3. **Culinary Delights:** **Themed Potluck Dinner.**  Assign each person a course or cuisine to prepare based on a fun theme (like "Around the World" or "Childhood Favorites").  Enjoy a delicious meal and share stories behind th

In [33]:
"""
Model 2
"""
pipe = pipeline("text-generation", model="marcuscedricridia/cursa-o1-7b-v1.2-normalize-false")
messages = [
    {"role": "user", "content": question},
]
pipe(messages, max_new_tokens=200)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

Device set to use cpu


[{'generated_text': [{'role': 'user',
    'content': 'Can you reccomend 5 different types of activies for a group of 4 people?'},
   {'role': 'assistant',
    'content': "Certainly! Here are five different types of activities that a group of four can enjoy together, each offering a unique experience:\n\n1. **Escape Room**: This is a perfect choice for a group that enjoys a bit of a challenge and teamwork. In an escape room, you're locked in a room and have to solve puzzles, find clues, and work together to escape within a limited time. It's not only fun but also great for fostering collaboration and problem-solving skills.\n\n2. **Board Game Night**: Gather around a table with some board games. This could be anything from classic games like Monopoly, Scrabble, or Clue, to more modern games like Ticket to Ride, Carcassonne, or Codenames. Board games are a great way to enjoy each other's company and engage in some friendly competition.\n\n3. **Culinary Class**: Cook up a storm together! 

In [30]:
"""
Model 3
"""
pipe = pipeline("text-generation", model="marcuscedricridia/Abus-7B-Instruct")
messages = [
    {"role": "user", "content": question},
]
pipe(messages, max_new_tokens=200)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cpu


[{'generated_text': [{'role': 'user',
    'content': 'Can you reccomend 5 different types of activies for a group of 4 people?'},
   {'role': 'assistant',
    'content': 'Certainly! Here are five different types of activities that a group of four people can enjoy together, catering to various interests and preferences:\n\n1. **Scavenger Hunt**: A scavenger hunt can be a fun and engaging way to explore a new place or even a familiar area. You can create a list of items, locations, or riddles to find. This activity encourages teamwork and gives everyone a chance to contribute and have fun.\n\n2. **Board Game Night**: Gather a mix of board games suitable for the group, such as card games, strategy games, or family games. This is a great way to spend an evening, with everyone participating and enjoying the social aspect of the game.\n\n3. **Cooking or Baking**: Cooking or baking together not only satisfies the taste buds but also provides a fun and interactive experience. Each person can t

# Massive (read note below)

In [36]:
"""
Model 1
"""
pipe = pipeline("text-generation", model="arcee-ai/Virtuoso-Lite")
messages = [
    {"role": "user", "content": question},
]
pipe(messages, max_new_tokens=128)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Device set to use cpu


KeyboardInterrupt: 

# Note for write-up (pt3)

-The massive models are extremely difficult to download and run on my laptop. They would take upwards of 15 minutes to download and generate a prompt. The previous model took over 30 minutes to download and respond. I even decreased the max tokens (200 to 128) but that didn't seem to speed it up that much. As a result, we can determine that the massive models are impractical for this project.

-In the future, the larger models should be configured on superior hardware. It would also be ideal if the device was NOT set to CPU. My Macbook is older and doesn't have the ram to accommodate these huge models.

-It appears the 0-6B is the sweetspot. Anything over 6B struggled to download and respond. I even relocated to a building (instead of my apartment) to use the campus wifi. This greatly helped the downloads, however, it still struggled with the large/massive models.

In [ ]:
"""
Model 2
"""
pipe = pipeline("text-generation", model="arcee-ai/Virtuoso-Lite")
messages = [
    {"role": "user", "content": question},
]
pipe(messages, max_new_tokens=128)

In [ ]:
"""
Model 3
"""
pipe = pipeline("text-generation", model="Aashraf995/Gemma-Evo-10B")
messages = [
    {"role": "user", "content": question},
]
pipe(messages, max_new_tokens=128)

# Prompt Engineering

Selected models:
-Ultra light (1)
-Small (2)
-Medium (2)

In [41]:
model_names = [
    "JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const",
    "google/gemma-2-2b-it",
    "Qwen/Qwen2.5-1.5B-Instruct",
]

import time, gc
import pandas as pd
from transformers import pipeline

def ask_llm(pipe, prompt, max_new_tokens=10):
    n_in = len(pipe.tokenizer(prompt)["input_ids"])
    start = time.time()
    out = pipe(prompt, max_new_tokens=max_new_tokens,
               do_sample=False, return_full_text=False)
    return {
        "text": out[0]["generated_text"],
        "seconds": round(time.time() - start, 2),
        "input_tokens": n_in,
    }

In [43]:

# 1. ZERO-SHOT:
#    Classify reviews without providing examples.
#
# 2. FEW-SHOT:
#    Classify reviews after providing three examples.
#
# HELD-OUT REVIEWS:
#    These are new customer reviews that were NOT included
#    in the examples provided to the model.
#
#    We use them to test whether the model can apply the
#    classification rules to unfamiliar inputs.
#

# ACCURACY FORMULA:
#    Accuracy = (Correct Predictions / Total Reviews) × 100

import pandas as pd

test_cases = [
    ("Hiking at a local trail.", "OUTDOORS"),
    ("Camping trip at a campsite.", "OUTDOORS"),
    ("Fishing at a local pond.", "OUTDOORS"),
    ("4 mile run around campus.", "FITNESS"),
    ("Leg day group at the gym.", "FITNESS"),
    ("Pickup basketball at the gym.", "FITNESS"),
    ("Trading Pokemon Cards.", "SOCIAL"),
    ("Bar crawl on campus", "SOCIAL"),
    ("Chess tournament.", "BOARD"),
    ("Catan game at legends.", "BOARD"),
    ("Group cake or brownies baking session", "FOOD"),
    ("group BBQ/grilling on campus", "FOOD"),
    ("Taking a dance class", "EDUCATION"),
    ("Study group for the diff-eq exam", "EDUCATION")
]

strategies = {
    "Zero-Shot": 0,
    "One-Shot": 1,
    "Few-Shot": 6,
    "Many-Shot": 12,
}

def sentiment_prompt(text, shots):
    """Create a sentiment classification prompt with optional examples."""

    examples = [
        ("Kayaking on the lake", "OUTDOORS"),
        ("Morning yoga session", "FITNESS"),
        ("Movie night with the hallmates", "SOCIAL"),
        ("Monopoly night", "BOARD"),
        ("Making homemade pizza together", "FOOD"),
        ("Resume workshop at the career center", "EDUCATION"),
        ("Bike ride along the river", "OUTDOORS"),
        ("Intramural volleyball", "FITNESS"),
        ("Karaoke at the student union", "SOCIAL"),
        ("Playing Risk with friends", "BOARD"),
        ("Tailgate potluck", "FOOD"),
        ("Calculus tutoring session", "EDUCATION"),
    ]

    instructions = (
        "Classify the activity into exactly one category.\n"
        "Return ONLY one of: OUTDOORS, FITNESS, SOCIAL, BOARD, FOOD, EDUCATION.\n\n"
    )

    demonstrations = ""
    for example_text, label in examples[:shots]:
        demonstrations += f"Activity: {example_text}\n"
        demonstrations += f"Label: {label}\n\n"

    return instructions + demonstrations + f"Activity: {text}\nLabel:"

# ------------------------------------------------------------
# STEP 3: TEST EACH STRATEGY
# ------------------------------------------------------------
results = []

for model_name in model_names:
    print(f"Loading {model_name} ...")
    pipe = pipeline("text-generation", model=model_name)

    for strategy, shots in strategies.items():
        for activity_text, expected in test_cases:
            prompt = sentiment_prompt(activity_text, shots)
            response = ask_llm(pipe, prompt, max_new_tokens=10)

            words = response["text"].strip().upper().split()
            predicted = words[0].strip(".,:") if words else ""

            results.append({
                "model": model_name,
                "strategy": strategy,
                "activity": activity_text,
                "expected": expected,
                "predicted": predicted,
                "correct": predicted == expected,
                "seconds": response["seconds"],
                "input_tokens": response["input_tokens"],
            })

    del pipe
    gc.collect()

results_df = pd.DataFrame(results)

accuracy = (results_df.groupby(["model", "strategy"])["correct"]
            .mean().mul(100).round(1).unstack("strategy"))
accuracy = accuracy[list(strategies.keys())]  # keep zero -> many order
print(accuracy)

summary = results_df.groupby(["model", "strategy"]).agg(
    correct_predictions=("correct", "sum"),
    total_input_tokens=("input_tokens", "sum"),
    total_seconds=("seconds", "sum"),
)
print(summary.to_string())

Loading JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const ...


Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading google/gemma-2-2b-it ...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cpu


Loading Qwen/Qwen2.5-1.5B-Instruct ...


Device set to use cpu
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


strategy                                    Zero-Shot  One-Shot  Few-Shot  \
model                                                                       
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const       28.6      35.7      21.4   
Qwen/Qwen2.5-1.5B-Instruct                       42.9      78.6      78.6   
google/gemma-2-2b-it                             35.7      78.6      78.6   

strategy                                    Many-Shot  
model                                                  
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const       28.6  
Qwen/Qwen2.5-1.5B-Instruct                       85.7  
google/gemma-2-2b-it                             85.7  
                                                      correct_predictions  total_input_tokens  total_seconds
model                                      strategy                                                         
JayHyeon/Qwen_0.5-cDPO_5e-7-3ep_0vpo_const Few-Shot                     3                1557          26.51
       

# Result

Based on this zero-shot, one-shot, few-shot, and many-shot test, we believe that the Qwen2.5-1.5B is the best model. It performed far better than the first model and is significantly quicker than the gemma model which performed similar. Ultimately, the model we use must be applicable for mobile devices which might have lower ram. This makes the gemma model difficult to justify. In this scenario, we value accuracy AND cost to use.